In [ ]:
import sys
sys.path.append("..")

import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from pathlib import Path
from PIL import Image

from src.config import METRICS_DIR, PLOTS_DIR, BEST_MODEL_PATH, IMG_SIZE

print("Setup abgeschlossen ✓")

In [ ]:
metrics_path = METRICS_DIR / "test_metrics.json"

if not metrics_path.exists():
    print("Keine Metriken gefunden.")
    print("Zuerst ausführen: python main.py --mode evaluate")
else:
    with open(metrics_path) as f:
        metrics = json.load(f)

    print("=" * 45)
    print("TEST-METRIKEN")
    print("=" * 45)
    print(f"  Accuracy    : {metrics['accuracy']   :.2%} "
          f"(95% CI: {metrics['accuracy_ci_low']:.2%} – "
          f"{metrics['accuracy_ci_high']:.2%})")
    print(f"  Precision   : {metrics['precision']  :.2%}")
    print(f"  Recall      : {metrics['recall']     :.2%}")
    print(f"  Specificity : {metrics['specificity']:.2%}")
    print(f"  F1-Score    : {metrics['f1']         :.2%}")
    print(f"  AUC         : {metrics['auc']        :.4f}")
    print(f"  AP Score    : {metrics['ap']         :.4f}")
    print(f"  Threshold   : {metrics['optimal_threshold']:.2f}")
    print("=" * 45)

In [ ]:
keys   = ["accuracy", "precision", "recall", "specificity", "f1"]
values = [metrics[k] * 100 for k in keys]
colors = ["#1D9E75" if v >= 90 else "#EF9F27" if v >= 80 else "#E24B4A"
          for v in values]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(keys, values, color=colors, edgecolor="white", height=0.5)

for bar, val in zip(bars, values):
    ax.text(bar.get_width() - 1.5, bar.get_y() + bar.get_height() / 2,
            f"{val:.1f}%", va="center", ha="right",
            color="white", fontsize=11, fontweight="bold")

ax.set_xlim(0, 105)
ax.set_xlabel("Wert (%)", fontsize=12)
ax.set_title("Modell-Metriken auf Testset", fontsize=14, fontweight="bold")
ax.axvline(x=90, color="gray", linestyle="--", linewidth=1,
           alpha=0.5, label="90% Schwelle")
ax.legend()
ax.grid(True, axis="x", alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS_DIR / "metrics_overview.png", dpi=150)
plt.show()

In [ ]:
history_path = METRICS_DIR / "training_history.json"

if not history_path.exists():
    print("Keine Training-History gefunden.")
else:
    with open(history_path) as f:
        history = json.load(f)

    epochs = range(1, len(history["train_loss"]) + 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(epochs, history["train_loss"],
             color="#378ADD", linewidth=2, label="Train Loss")
    ax1.plot(epochs, history["val_loss"],
             color="#E24B4A", linewidth=2, label="Val Loss")
    ax1.set_title("Loss",    fontsize=13, fontweight="bold")
    ax1.set_xlabel("Epoche")
    ax1.set_ylabel("Loss")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(epochs, [a * 100 for a in history["train_acc"]],
             color="#378ADD", linewidth=2, label="Train Accuracy")
    ax2.plot(epochs, [a * 100 for a in history["val_acc"]],
             color="#E24B4A", linewidth=2, label="Val Accuracy")
    ax2.set_title("Accuracy", fontsize=13, fontweight="bold")
    ax2.set_xlabel("Epoche")
    ax2.set_ylabel("Accuracy (%)")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.suptitle("Training History", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "training_history.png", dpi=150)
    plt.show()

    best_epoch = np.argmax(history["val_acc"]) + 1
    best_acc   = max(history["val_acc"]) * 100
    print(f"Beste Epoche  : {best_epoch}")
    print(f"Beste Val-Acc : {best_acc:.2f}%")

In [ ]:
saved_plots = {
    "Confusion Matrix"    : PLOTS_DIR / "confusion_matrix.png",
    "ROC Kurve"           : PLOTS_DIR / "roc_curve.png",
    "Precision-Recall"    : PLOTS_DIR / "precision_recall_curve.png",
    "Grad-CAM"            : PLOTS_DIR / "gradcam.png",
    "Falsch klassifiziert": PLOTS_DIR / "misclassified.png",
}

for title, path in saved_plots.items():
    if path.exists():
        fig, ax = plt.subplots(figsize=(10, 7))
        ax.imshow(Image.open(path))
        ax.set_title(title, fontsize=14, fontweight="bold")
        ax.axis("off")
        plt.tight_layout()
        plt.show()
    else:
        print(f"Nicht gefunden: {path.name} → evaluate zuerst ausführen")

In [ ]:
from src.model   import load_model
from src.predict import GradCAM, get_last_conv_layer, preprocess_image

device     = torch.device("cpu")
model      = load_model(BEST_MODEL_PATH, device)
cam        = GradCAM(model, get_last_conv_layer(model))

test_images = list(
    Path("../data/processed/test/infected").glob("*.png")
)[:4]

if not test_images:
    print("Keine Testbilder gefunden.")
else:
    fig, axes = plt.subplots(len(test_images), 3,
                             figsize=(10, len(test_images) * 3))

    for i, img_path in enumerate(test_images):
        tensor, original = preprocess_image(img_path, img_size=IMG_SIZE)

        with torch.enable_grad():
            heatmap = cam(tensor.to(device), class_idx=1)

        overlay = GradCAM.overlay(original, heatmap)

        axes[i, 0].imshow(original)
        axes[i, 0].set_title("Original",  fontsize=9)
        axes[i, 0].axis("off")

        axes[i, 1].imshow(heatmap, cmap="jet")
        axes[i, 1].set_title("Heatmap",   fontsize=9)
        axes[i, 1].axis("off")

        axes[i, 2].imshow(overlay)
        axes[i, 2].set_title("Überlagert", fontsize=9)
        axes[i, 2].axis("off")

    plt.suptitle("Grad-CAM – eigene Testbilder",
                 fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "gradcam_custom.png", dpi=150)
    plt.show()

In [ ]:
print("=" * 50)
print("ZUSAMMENFASSUNG FÜR ETH-PRÄSENTATION")
print("=" * 50)
print(f"  Accuracy    : {metrics['accuracy']   :.2%}")
print(f"  Recall      : {metrics['recall']     :.2%}  ← infizierte Zellen erkannt")
print(f"  Specificity : {metrics['specificity']:.2%}  ← gesunde Zellen korrekt")
print(f"  F1-Score    : {metrics['f1']         :.2%}")
print(f"  AUC         : {metrics['auc']        :.4f}")
print(f"  AP Score    : {metrics['ap']         :.4f}")
print()
print(f"  95% Konfidenzintervall:")
print(f"  {metrics['accuracy_ci_low']:.2%} – {metrics['accuracy_ci_high']:.2%}")
print()
print(f"  Optimaler Schwellenwert: {metrics['optimal_threshold']:.2f}")
print("=" * 50)